# Modelo predictivo de viviendas adaptado a Bogotá

En este trabajo voy a usar **Regresión Lineal** para estimar el precio de una vivienda y **Regresión Logística** para estimar si una vivienda tiene un precio alto o no.

**Importante:** el archivo `Housing.csv` no contiene una columna de ciudad y no es un dataset específico de Bogotá. Por eso, la adaptación a Bogotá se hace en la interpretación y en el ejemplo final. Para obtener resultados reales de Bogotá se necesitaría un dataset de viviendas de Bogotá.

In [ ]:
# Importo pandas para leer y organizar los datos.
import pandas as pd
# Importo numpy para realizar cálculos numéricos.
import numpy as np
# Importo matplotlib para mostrar gráficos.
import matplotlib.pyplot as plt
# Importo train_test_split para separar los datos de entrenamiento y prueba.
from sklearn.model_selection import train_test_split
# Importo ColumnTransformer para preparar columnas de diferentes tipos.
from sklearn.compose import ColumnTransformer
# Importo OneHotEncoder para convertir textos en valores que el modelo pueda utilizar.
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Importo la regresión lineal para predecir el precio de la vivienda.
from sklearn.linear_model import LinearRegression, LogisticRegression
# Importo las métricas para revisar el resultado de los modelos.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix, classification_report
# Importo Pipeline para unir la preparación de los datos con cada modelo.
from sklearn.pipeline import Pipeline

In [ ]:
# Leo el archivo Housing.csv que está cargado en Google Colab.
datos = pd.read_csv('Housing.csv')
# Muestro las primeras filas para comprobar que el archivo se cargó correctamente.
datos.head()

In [ ]:
# Muestro el tamaño del dataset para saber cuántos registros y columnas tiene.
print('Tamaño del dataset:', datos.shape)
# Muestro los nombres de las columnas para conocer las características disponibles.
print('Columnas:', list(datos.columns))
# Reviso si existen datos vacíos que puedan afectar el entrenamiento.
print('\nValores faltantes:')
print(datos.isnull().sum())

In [ ]:
# Elimino los registros que tengan datos faltantes para trabajar con información completa.
datos = datos.dropna()
# Separo las características de las viviendas de la columna price.
X = datos.drop(columns=['price'])
# Guardo price porque será el valor que la regresión lineal intentará predecir.
y = datos['price']
# Identifico las columnas que contienen texto.
columnas_texto = X.select_dtypes(include='object').columns
# Identifico las columnas que contienen números.
columnas_numericas = X.select_dtypes(exclude='object').columns
# Muestro las columnas de cada tipo para comprobar la preparación.
print('Columnas de texto:', list(columnas_texto))
print('Columnas numéricas:', list(columnas_numericas))

In [ ]:
# Creo el preparador para convertir las columnas de texto y conservar las numéricas.
preparador_lineal = ColumnTransformer([
    ('texto', OneHotEncoder(handle_unknown='ignore'), columnas_texto)
], remainder='passthrough')
# Separo el 80% de los datos para entrenar y el 20% para probar.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Muestro la cantidad de datos que se usarán en cada parte.
print('Datos de entrenamiento:', X_train.shape)
print('Datos de prueba:', X_test.shape)

## 1. Regresión Lineal – precio de la vivienda

La regresión lineal busca estimar un **precio numérico** usando las características de la vivienda.

In [ ]:
# Creo el modelo de regresión lineal para estimar el precio de las viviendas.
modelo_lineal = Pipeline([
    ('preparacion', preparador_lineal),
    ('modelo', LinearRegression())
])
# Entreno el modelo con los datos de entrenamiento.
modelo_lineal.fit(X_train, y_train)
# Genero predicciones de precio para las viviendas de prueba.
pred_lineal = modelo_lineal.predict(X_test)
# Calculo el error promedio de las predicciones.
mae_lineal = mean_absolute_error(y_test, pred_lineal)
# Calculo el error RMSE para dar mayor importancia a los errores grandes.
rmse_lineal = np.sqrt(mean_squared_error(y_test, pred_lineal))
# Calculo R2 para saber qué tan bien explica el modelo los precios.
r2_lineal = r2_score(y_test, pred_lineal)
# Muestro los resultados de la regresión lineal.
print('REGRESIÓN LINEAL')
print('MAE:', mae_lineal)
print('RMSE:', rmse_lineal)
print('R2:', r2_lineal)

## 2. Regresión Logística – precio alto o no

La regresión logística **no debe usarse para predecir directamente el precio**, porque el precio es un número continuo. Para usarla en esta tarea, convierto el problema en una clasificación: **1 = precio alto** y **0 = precio no alto**, tomando como referencia la mediana de los precios del dataset.

In [ ]:
# Calculo la mediana de los precios para establecer el punto que separará los precios altos.
mediana_precio = y.median()
# Creo una nueva variable donde 1 significa precio alto y 0 significa precio no alto.
y_logistica = (y >= mediana_precio).astype(int)
# Separo los datos para entrenar y probar la regresión logística usando la misma división.
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y_logistica, test_size=0.2, random_state=42, stratify=y_logistica)
# Creo el preparador para convertir las categorías y escalar los números.
preparador_logistica = ColumnTransformer([
    ('texto', OneHotEncoder(handle_unknown='ignore'), columnas_texto),
    ('numeros', StandardScaler(), columnas_numericas)
])
# Creo el modelo de regresión logística para clasificar las viviendas.
modelo_logistica = Pipeline([
    ('preparacion', preparador_logistica),
    ('modelo', LogisticRegression(max_iter=1000))
])
# Entreno la regresión logística con los datos preparados.
modelo_logistica.fit(X_train_log, y_train_log)
# Predigo si las viviendas de prueba tienen un precio alto o no.
pred_logistica = modelo_logistica.predict(X_test_log)
# Obtengo la probabilidad de que cada vivienda pertenezca al grupo de precio alto.
probabilidad_precio_alto = modelo_logistica.predict_proba(X_test_log)[:, 1]
# Calculo el porcentaje de clasificaciones correctas.
exactitud = accuracy_score(y_test_log, pred_logistica)
# Muestro la mediana que se utilizó como referencia.
print('Mediana del precio:', mediana_precio)
# Muestro la exactitud obtenida por la regresión logística.
print('Exactitud:', exactitud)

In [ ]:
# Muestro la matriz de confusión para revisar los aciertos y errores de clasificación.
matriz = confusion_matrix(y_test_log, pred_logistica)
# Muestro la matriz de confusión.
print('Matriz de confusión:')
print(matriz)
# Muestro un resumen de precisión, aciertos y errores del modelo.
print('\nReporte de clasificación:')
print(classification_report(y_test_log, pred_logistica))

## 3. Ejemplo adaptado a Bogotá

El siguiente ejemplo representa una vivienda hipotética en Bogotá. **La predicción se expresa en las mismas unidades monetarias del `Housing.csv`; no se convierte automáticamente a pesos colombianos porque el archivo no contiene una tasa de conversión ni datos reales de Bogotá.**

In [ ]:
# Creo una vivienda de ejemplo pensando en un caso de Bogotá.
casa_bogota = pd.DataFrame({
    'area': [1500],
    'bedrooms': [3],
    'bathrooms': [2],
    'stories': [2],
    'mainroad': ['yes'],
    'guestroom': ['no'],
    'basement': ['yes'],
    'hotwaterheating': ['no'],
    'airconditioning': ['yes'],
    'parking': [2],
    'prefarea': ['yes'],
    'furnishingstatus': ['semi-furnished']
})
# Estimo el precio de la vivienda con la regresión lineal.
precio_estimado = modelo_lineal.predict(casa_bogota)[0]
# Calculo la probabilidad de que la vivienda tenga un precio alto.
probabilidad_alta = modelo_logistica.predict_proba(casa_bogota)[0, 1]
# Convierto la probabilidad en una clasificación sencilla.
clasificacion = 'Precio alto' if probabilidad_alta >= 0.5 else 'Precio no alto'
# Muestro el precio estimado por el modelo lineal.
print(f'Precio estimado: {precio_estimado:,.0f}')
# Muestro la probabilidad calculada por la regresión logística.
print(f'Probabilidad de precio alto: {probabilidad_alta:.2%}')
# Muestro la clasificación final de la vivienda.
print('Clasificación:', clasificacion)

## Análisis

La **Regresión Lineal** es la más adecuada cuando quiero obtener un precio estimado porque entrega un valor numérico. La **Regresión Logística** sirve para responder otra pregunta: si una vivienda tiene una probabilidad alta de pertenecer al grupo de precios altos. En este trabajo, la clasificación se define usando la mediana del dataset.

La adaptación a Bogotá es principalmente de contexto, ya que `Housing.csv` no contiene datos específicos de Bogotá. Por eso, los resultados no deben presentarse como precios reales del mercado bogotano. Para una predicción real de casas en Bogotá sería necesario utilizar datos de viviendas de Bogotá.